# 01 — Del audio a sus features
## 1. Imports y comprobación de entorno
Un audio digital es una secuencia de números. Vamos a convertirla en matrices que
resumen cómo cambia su contenido frecuencial. Ejecuta las seis secciones en orden,
tras configurar `AUDIO_PATH` en la sección 2. Basta una CPU y un fragmento de unos segundos.

Instala las dependencias de `requirements.txt` en el entorno del kernel de Jupyter.
Una combinación reproducible para este notebook es Python 3.11, `torch==2.8.0`,
`torchaudio==2.8.0` y `soundfile`, además de librosa, numpy y matplotlib.
Torch y torchaudio deben tener versiones compatibles. Las versiones recientes de
torchaudio delegan la lectura en TorchCodec: si usas esa vía, necesitas también
TorchCodec y sus dependencias de decodificación compatibles.

Usaremos **torchaudio** para cargar y transformar, **torch** para tensores,
**numpy** para coordenadas y **matplotlib** para gráficos. Importamos **librosa**
como alternativa habitual para análisis musical; no hace falta CUDA.
Fuentes del repositorio: [torchaudio](https://pytorch.org/audio/stable/index.html)
y [librosa](https://librosa.org/doc/latest/index.html).

In [ ]:
from pathlib import Path
import torch
import torchaudio
import librosa
import numpy as np
import matplotlib
import matplotlib.pyplot as plt

for modulo in (torch, torchaudio, librosa, np, matplotlib):
    print(f"{modulo.__name__}: {modulo.__version__}")
print(f"CUDA disponible: {torch.cuda.is_available()} (trabajaremos en CPU)")
plt.rcParams.update({"figure.figsize": (11, 4), "axes.grid": False})

## 2. Carga de un audio propio
**Edita `AUDIO_PATH` para apuntar a tu archivo local**, preferiblemente un WAV corto.
El audio **NO se versiona**: `.gitignore` excluye WAV, FLAC, MP3 y otros formatos.
Cada persona utiliza el suyo; este notebook no descarga ni incluye grabaciones.
Una ruta absoluta evita depender del directorio desde el que arrancaste Jupyter.

`torchaudio.load` devuelve un tensor `[canales, muestras]` y la frecuencia de
muestreo (muestras por segundo, Hz). La duración es `muestras / sample_rate`.
Promediamos los canales para obtener mono: se pierde la información espacial y
los canales en oposición de fase pueden cancelarse. No cambiamos el sample rate.
Alternativa: `librosa.load(..., sr=None, mono=False)` conserva el sample rate original.

In [ ]:
AUDIO_PATH = Path.home() / "Music" / "mi_audio.wav"  # Sustituye por tu archivo.
AUDIO_PATH = AUDIO_PATH.expanduser()
if not AUDIO_PATH.is_file():
    raise FileNotFoundError(f"Configura AUDIO_PATH con un audio local existente: {AUDIO_PATH}")

waveform, sample_rate = torchaudio.load(str(AUDIO_PATH))
waveform = waveform.to(device="cpu", dtype=torch.float32)
canales, muestras = waveform.shape
if sample_rate <= 0 or muestras == 0 or not torch.isfinite(waveform).all():
    raise ValueError("El audio debe contener muestras finitas y un sample rate positivo.")
duracion = muestras / sample_rate
print(f"Frecuencia de muestreo: {sample_rate} Hz")
print(f"Canales originales: {canales}")
print(f"Duración: {duracion:.3f} s; muestras por canal: {muestras}")
waveform = waveform.mean(dim=0, keepdim=True)
print(f"Tensor mono [canales, muestras]: {tuple(waveform.shape)}")

## 3. Forma de onda
El eje **X** es el tiempo en segundos; cada muestra está separada `1 / sample_rate`
segundos de la siguiente. El eje **Y** es la amplitud digital con signo, normalmente
cercana al intervalo −1 a 1 en audio PCM convertido a flotante; no es presión sonora
calibrada ni volumen percibido. La curva permite localizar ataques, pausas y cambios
de amplitud, pero no separa las frecuencias que suenan simultáneamente.

In [ ]:
tiempo = np.arange(muestras) / sample_rate
fig, ax = plt.subplots()
ax.plot(tiempo, waveform[0].numpy(), linewidth=0.6)
ax.set(xlabel="Tiempo (s)", ylabel="Amplitud digital", title="Forma de onda — mono")
fig.tight_layout()
plt.show()

## 4. Espectrograma: STFT y magnitud en dB
La STFT aplica una FFT a ventanas sucesivas. Usamos una ventana Hann de 1024 muestras
y un salto de 256: la ventana dura `1024 / sample_rate` segundos y las columnas se
separan `256 / sample_rate` segundos. Una ventana mayor mejora la resolución
frecuencial a costa de localizar peor los cambios rápidos.

El eje **X** indica el tiempo del centro de cada ventana; el **Y**, frecuencia en Hz
hasta Nyquist (`sample_rate / 2`). El **color** representa magnitud en dB relativa al
máximo del archivo: `20 log10(magnitud / máximo)`. Cero dB es el máximo, no silencio
ni un nivel acústico absoluto. Mostramos un rango de 80 dB y un suelo numérico para
evitar `log(0)`. En silencio total no hay un pico de referencia físico.

El relleno con ceros permite analizar también clips menores que una ventana; los
bordes quedan afectados por ese relleno. Alternativa: `librosa.stft` y
`librosa.amplitude_to_db` con los mismos parámetros.

In [ ]:
N_FFT = 1024
HOP_LENGTH = 256
N_MELS = 64
N_MFCC = 13

stft = torch.stft(
    waveform[0], n_fft=N_FFT, hop_length=HOP_LENGTH,
    window=torch.hann_window(N_FFT), center=True,
    pad_mode="constant", return_complex=True,
)
magnitud = stft.abs()
epsilon = 1e-10
referencia = magnitud.max().clamp_min(epsilon)
espectrograma_db = 20 * torch.log10(magnitud.clamp_min(epsilon) / referencia)
espectrograma_db = espectrograma_db.clamp(min=-80)
tiempos_stft = np.arange(magnitud.shape[1]) * HOP_LENGTH / sample_rate
frecuencias = np.fft.rfftfreq(N_FFT, d=1 / sample_rate)
print(f"Magnitud [frecuencias, ventanas]: {tuple(magnitud.shape)}")
print(f"Separación frecuencial: {sample_rate / N_FFT:.2f} Hz")

fig, ax = plt.subplots()
im = ax.pcolormesh(tiempos_stft, frecuencias, espectrograma_db.numpy(),
                   shading="nearest", cmap="magma", vmin=-80, vmax=0)
ax.set(xlabel="Tiempo (s)", ylabel="Frecuencia (Hz)", title="STFT — magnitud relativa")
fig.colorbar(im, ax=ax, label="Magnitud (dB respecto al máximo)")
fig.tight_layout()
plt.show()

## 5. Mel-espectrograma
Agrupamos la potencia de la STFT con 64 filtros triangulares espaciados en la escala
mel, una aproximación a la percepción de altura. Hay más detalle en graves y bandas
más anchas en agudos que con intervalos uniformes en Hz. Esto reduce la dimensión
frecuencial y pierde detalle: no es simplemente cambiar las etiquetas del gráfico.

El eje **X** sigue siendo tiempo; el **Y** es el índice de banda mel (no Hz).
El **color** expresa potencia en dB respecto al máximo de esta matriz:
`10 log10(potencia / máximo)`. Usamos 10 porque es potencia, frente a 20 para magnitud.
Las referencias de ambos gráficos son independientes; sus colores no comparan niveles
absolutos. Alternativa: `librosa.feature.melspectrogram` y `librosa.power_to_db`.

In [ ]:
parametros_mel = dict(
    n_fft=N_FFT, hop_length=HOP_LENGTH, n_mels=N_MELS,
    f_min=0.0, f_max=sample_rate / 2, power=2.0,
    center=True, pad_mode="constant", norm="slaney", mel_scale="slaney",
)
transformada_mel = torchaudio.transforms.MelSpectrogram(
    sample_rate=sample_rate, **parametros_mel,
)
mel_potencia = transformada_mel(waveform)[0]
referencia_mel = mel_potencia.max().clamp_min(epsilon)
mel_db = 10 * torch.log10(mel_potencia.clamp_min(epsilon) / referencia_mel)
mel_db = mel_db.clamp(min=-80)
tiempos_mel = np.arange(mel_potencia.shape[1]) * HOP_LENGTH / sample_rate
print(f"Mel [bandas, ventanas]: {tuple(mel_potencia.shape)}")

fig, ax = plt.subplots()
im = ax.pcolormesh(tiempos_mel, np.arange(N_MELS), mel_db.numpy(),
                   shading="nearest", cmap="magma", vmin=-80, vmax=0)
ax.set(xlabel="Tiempo (s)", ylabel="Índice de banda mel", title="Mel-espectrograma")
fig.colorbar(im, ax=ax, label="Potencia (dB respecto al máximo)")
fig.tight_layout()
plt.show()

## 6. MFCC: una representación compacta del timbre
Los MFCC aplican una transformada discreta del coseno (DCT) al mel-espectrograma
logarítmico y conservan aquí 13 coeficientes. Resumen la envolvente espectral y se
usan como features en reconocimiento de voz, clasificación de sonidos y comparación
de timbres. No conservan la fase ni permiten reconstruir exactamente el audio.

El eje **X** es tiempo, el **Y** es el índice del coeficiente (no una frecuencia),
y el **color** su valor, que puede ser positivo o negativo. Los coeficientes bajos
capturan variaciones amplias de la envolvente; el coeficiente 0 es sensible al nivel.
Usamos `log_mels=True`: torchaudio calcula el logaritmo natural de la potencia mel
antes de la DCT, no reutiliza el gráfico en dB normalizado de arriba. Alternativa:
`librosa.feature.mfcc`, igualando convenciones y parámetros para comparar resultados.

Cada columna es ahora un vector de 13 números. Prueba dos sonidos con distinto
timbre: ¿qué diferencias ves en la forma de onda, las bandas mel y los MFCC?

In [ ]:
transformada_mfcc = torchaudio.transforms.MFCC(
    sample_rate=sample_rate, n_mfcc=N_MFCC,
    dct_type=2, norm="ortho", log_mels=True, melkwargs=parametros_mel,
)
mfcc = transformada_mfcc(waveform)[0]
tiempos_mfcc = np.arange(mfcc.shape[1]) * HOP_LENGTH / sample_rate
limite = max(float(mfcc.abs().max()), 1e-6)
print(f"MFCC [coeficientes, ventanas]: {tuple(mfcc.shape)}")

fig, ax = plt.subplots()
im = ax.pcolormesh(tiempos_mfcc, np.arange(N_MFCC), mfcc.numpy(),
                   shading="nearest", cmap="RdBu_r", vmin=-limite, vmax=limite)
ax.set(xlabel="Tiempo (s)", ylabel="Índice de coeficiente MFCC", title="MFCC — 13 features por ventana")
ax.set_yticks(np.arange(N_MFCC))
fig.colorbar(im, ax=ax, label="Valor del coeficiente")
fig.tight_layout()
plt.show()